<a href="https://colab.research.google.com/github/skfaisalahmed456/Q-A-bot/blob/main/Document_QA_RAG_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Document Q&A RAG Bot

**Retrieval-Augmented Generation (RAG)** bot built with LangChain, FAISS, HuggingFace embeddings (`all-MiniLM-L6-v2`), and **Google Gemini 2.5 Flash**.

The bot answers questions **only** from the content of the uploaded documents (PDF / DOCX / TXT). If the answer is not present in the documents, it replies:

> "I couldn't find this information in the uploaded documents."

Every answer is displayed with a citation in the form `filename Page N`.

**How to use this notebook:**
- Run the cells **in order**, top to bottom.
- Cells 1–12 form the **Indexing Pipeline** (run once per document set).
- Cells 13–18 form the **Querying Pipeline** (can be re-run independently after indexing is done).


## Cell 1 — Install Libraries

In [1]:
# Install all required libraries for the RAG pipeline.
#
# NOTE: langchain 1.x (released 2025) moved several pieces out of the core
# "langchain" package:
#   - Document            -> now lives in langchain-core
#   - RecursiveCharacterTextSplitter -> now lives in langchain-text-splitters
#   - PromptTemplate      -> now lives in langchain-core
#   - RetrievalQA         -> moved to the new "langchain-classic" package
#       (legacy chains were split out of "langchain" itself in v1.0)
# langchain-core and langchain-text-splitters install automatically as
# dependencies of "langchain", but langchain-classic must be installed
# explicitly, which is why it is listed below.
#
# Restart the runtime if prompted after installation (Colab sometimes requires this).
!pip install -q langchain langchain-classic langchain-community langchain-google-genai faiss-cpu sentence-transformers pypdf python-docx google-generativeai

print("All required libraries installed successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All required libraries installed successfully.


## Cell 2 — Upload Documents

In [2]:
import os

# Folder where source documents (PDF, DOCX, TXT) are stored
DOCS_DIR = "docs"
os.makedirs(DOCS_DIR, exist_ok=True)

# Detect whether we are running inside Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Please select 4-5 documents to upload (PDF, DOCX, and/or TXT).")
    uploaded = files.upload()
    for filename, file_bytes in uploaded.items():
        destination_path = os.path.join(DOCS_DIR, filename)
        try:
            with open(destination_path, "wb") as f:
                f.write(file_bytes)
            print(f"Saved: {destination_path}")
        except Exception as e:
            print(f"Error saving {filename}: {e}")
else:
    print(f"Not running in Google Colab.")
    print(f"Please manually place your PDF/DOCX/TXT files inside the '{DOCS_DIR}/' folder.")

print("\nDocuments currently available in docs/:")
for f in sorted(os.listdir(DOCS_DIR)):
    if not f.startswith("."):
        print(f"  - {f}")


Please select 4-5 documents to upload (PDF, DOCX, and/or TXT).


Saving Cloud_Computing.txt to Cloud_Computing.txt
Saving Database.docx to Database.docx
Saving Machine_Learning.pdf to Machine_Learning.pdf
Saving AI.pdf to AI.pdf
Saving Python.pdf to Python.pdf
Saved: docs/Cloud_Computing.txt
Saved: docs/Database.docx
Saved: docs/Machine_Learning.pdf
Saved: docs/AI.pdf
Saved: docs/Python.pdf

Documents currently available in docs/:
  - AI.pdf
  - Cloud_Computing.txt
  - Database.docx
  - Machine_Learning.pdf
  - Python.pdf


## Cell 3 — Imports

In [5]:
# ─── Fix NLTK circular import (Colab environment issue) ───────────────────────
import sys

# Purge any partially-initialized nltk from the module cache before anything
# imports langchain_text_splitters (which triggers nltk's downloader at import
# time and can hit a circular-import race in Colab).
for mod in list(sys.modules.keys()):
    if mod == "nltk" or mod.startswith("nltk."):
        del sys.modules[mod]
# ──────────────────────────────────────────────────────────────────────────────

import os
import glob
import getpass

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai

from docx import Document as DocxDocument

DOCS_DIR        = "docs"
VECTORSTORE_DIR = "vectorstore"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEMINI_MODEL_NAME    = "gemini-2.5-flash"

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(VECTORSTORE_DIR, exist_ok=True)

print("All libraries imported successfully.")

/tmp/ipykernel_2212/4107852486.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All libraries imported successfully.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Cell 4 — Configure Google Gemini API

In [6]:
import os
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai

GEMINI_MODEL_NAME = "gemini-2.5-flash"

# Try to read the API key from the environment first (works with .env-loaded
# environments locally). If not found, prompt securely (safe for Colab/Jupyter).
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass.getpass("Enter your Google Gemini API key: ")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

try:
    # Configure the official Google Generative AI SDK
    genai.configure(api_key=GOOGLE_API_KEY)

    # Initialize the Gemini 2.5 Flash chat model through LangChain
    llm = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL_NAME,
        temperature=0,
        google_api_key=GOOGLE_API_KEY,
    )

    # Quick sanity check to confirm the API key and connection work
    test_response = llm.invoke("Reply with the single word: OK")
    print("Gemini API connection successful.")
    print(f"Model: {GEMINI_MODEL_NAME}")
    print(f"Test response: {test_response.content}")

except Exception as e:
    print(f"Error connecting to Gemini API: {e}")
    print("Please check that your API key is valid and has access to Gemini models.")


Enter your Google Gemini API key: ··········
Gemini API connection successful.
Model: gemini-2.5-flash
Test response: OK


## Cell 5 — Load PDF Documents

In [7]:
import os, hashlib

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

DOCS_DIR = "docs"
seen_hashes = {}
removed = []

for filename in sorted(os.listdir(DOCS_DIR)):
    filepath = os.path.join(DOCS_DIR, filename)
    if not os.path.isfile(filepath):
        continue
    h = file_hash(filepath)
    if h in seen_hashes:
        os.remove(filepath)
        removed.append(filename)
    else:
        seen_hashes[h] = filename

print("Removed duplicate files:", removed if removed else "none found")
print("Remaining files:", sorted(os.listdir(DOCS_DIR)))

Removed duplicate files: none found
Remaining files: ['AI.pdf', 'Cloud_Computing.txt', 'Database.docx', 'Machine_Learning.pdf', 'Python.pdf']


## Cell 6 — Load DOCX Documents

In [8]:
import os
import hashlib

def file_hash(path):
    hash_md5 = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

DOCS_DIR = "docs"

seen_hashes = {}
removed = []

for filename in sorted(os.listdir(DOCS_DIR)):
    filepath = os.path.join(DOCS_DIR, filename)

    if not os.path.isfile(filepath):
        continue

    h = file_hash(filepath)

    if h in seen_hashes:
        os.remove(filepath)
        removed.append(filename)
    else:
        seen_hashes[h] = filename

print("Removed duplicate files:", removed if removed else "None")
print("Remaining files:", sorted(os.listdir(DOCS_DIR)))

Removed duplicate files: None
Remaining files: ['AI.pdf', 'Cloud_Computing.txt', 'Database.docx', 'Machine_Learning.pdf', 'Python.pdf']


## Cell 7 — Load TXT Documents

In [9]:
import os
from langchain_core.documents import Document

# Folder containing your documents
DOCS_DIR = "docs"

def load_txt_documents(docs_dir):
    """
    Load all TXT files from the given directory.
    """

    txt_documents = []

    # Check if folder exists
    if not os.path.exists(docs_dir):
        raise FileNotFoundError(f"Folder '{docs_dir}' not found.")

    # Find all txt files
    txt_files = sorted(
        f for f in os.listdir(docs_dir)
        if f.lower().endswith(".txt")
    )

    if not txt_files:
        print("No TXT files found.")
        return txt_documents

    for filename in txt_files:

        filepath = os.path.join(docs_dir, filename)

        try:
            # Try UTF-8 first
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    text = f.read()
            except UnicodeDecodeError:
                # Fallback encoding
                with open(filepath, "r", encoding="latin-1") as f:
                    text = f.read()

            document = Document(
                page_content=text,
                metadata={
                    "filename": filename,
                    "page": "N/A",
                    "doc_type": "txt",
                    "source": filepath,
                },
            )

            txt_documents.append(document)

            print(f"✓ Loaded: {filename}")

        except Exception as e:
            print(f"❌ Error loading {filename}: {e}")

    return txt_documents


# Load TXT documents
txt_documents = load_txt_documents(DOCS_DIR)

print(f"\nTotal TXT documents loaded: {len(txt_documents)}")

✓ Loaded: Cloud_Computing.txt

Total TXT documents loaded: 1


In [10]:
import os

DOCS_DIR = "docs"

print(os.listdir(DOCS_DIR))

['Database.docx', 'Python.pdf', 'Cloud_Computing.txt', 'AI.pdf', 'Machine_Learning.pdf']


In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from docx import Document as DocxDocument

DOCS_DIR = "docs"

# ─── 1. PDF Loader ────────────────────────────────────────────────────────────
pdf_documents = []
pdf_files = sorted(f for f in os.listdir(DOCS_DIR) if f.lower().endswith(".pdf"))

for filename in pdf_files:
    filepath = os.path.join(DOCS_DIR, filename)
    try:
        loader = PyPDFLoader(filepath)
        pages  = loader.load()
        # Normalize metadata so every doc has a consistent 'filename' key
        for page in pages:
            page.metadata["filename"] = filename
            page.metadata["doc_type"] = "pdf"
        pdf_documents.extend(pages)
        print(f"✓ PDF  | {filename:40s} → {len(pages)} pages")
    except Exception as e:
        print(f"✗ PDF  | {filename}: {e}")

# ─── 2. DOCX Loader ───────────────────────────────────────────────────────────
docx_documents = []
docx_files = sorted(f for f in os.listdir(DOCS_DIR) if f.lower().endswith(".docx"))

for filename in docx_files:
    filepath = os.path.join(DOCS_DIR, filename)
    try:
        raw_doc   = DocxDocument(filepath)
        full_text = "\n".join(p.text for p in raw_doc.paragraphs if p.text.strip())
        # Chunk into ~1-page pseudo-pages of 3 000 chars so metadata is granular
        chunk_size = 3000
        for i, start in enumerate(range(0, len(full_text), chunk_size)):
            docx_documents.append(Document(
                page_content=full_text[start : start + chunk_size],
                metadata={"filename": filename, "page": i + 1, "doc_type": "docx", "source": filepath},
            ))
        print(f"✓ DOCX | {filename:40s} → {len(docx_documents)} pseudo-pages")
    except Exception as e:
        print(f"✗ DOCX | {filename}: {e}")

# ─── 3. TXT Loader ────────────────────────────────────────────────────────────
txt_documents = []
txt_files = sorted(f for f in os.listdir(DOCS_DIR) if f.lower().endswith(".txt"))

for filename in txt_files:
    filepath = os.path.join(DOCS_DIR, filename)
    try:
        try:
            text = open(filepath, encoding="utf-8").read()
        except UnicodeDecodeError:
            text = open(filepath, encoding="latin-1").read()
        txt_documents.append(Document(
            page_content=text,
            metadata={"filename": filename, "page": "N/A", "doc_type": "txt", "source": filepath},
        ))
        print(f"✓ TXT  | {filename:40s} → 1 document")
    except Exception as e:
        print(f"✗ TXT  | {filename}: {e}")

# ─── 4. Summary ───────────────────────────────────────────────────────────────
all_documents = pdf_documents + docx_documents + txt_documents

print("\n" + "=" * 55)
print(f"  PDF pages       : {len(pdf_documents)}")
print(f"  DOCX pseudo-pgs : {len(docx_documents)}")
print(f"  TXT files       : {len(txt_documents)}")
print(f"  TOTAL           : {len(all_documents)}")
print("=" * 55)

✓ PDF  | AI.pdf                                   → 4 pages
✓ PDF  | Machine_Learning.pdf                     → 4 pages
✓ PDF  | Python.pdf                               → 4 pages
✓ DOCX | Database.docx                            → 3 pseudo-pages
✓ TXT  | Cloud_Computing.txt                      → 1 document

  PDF pages       : 12
  DOCX pseudo-pgs : 3
  TXT files       : 1
  TOTAL           : 16


## Cell 8 — Merge Documents

In [13]:
# Combine all loaded documents (PDF pages + DOCX pseudo-pages + TXT files)
# into a single list for downstream chunking and embedding.
all_documents = pdf_documents + docx_documents + txt_documents

print("=" * 50)
print("Document Loading Summary")
print("=" * 50)
print(f"PDF pages         : {len(pdf_documents)}")
print(f"DOCX pseudo-pages : {len(docx_documents)}")
print(f"TXT files         : {len(txt_documents)}")
print("-" * 50)
print(f"Total documents merged: {len(all_documents)}")

if len(all_documents) == 0:
    print("\nWARNING: No documents were loaded. Please check the 'docs/' folder")
    print("and re-run Cell 2 to upload PDF, DOCX, or TXT files.")
else:
    unique_files = sorted({doc.metadata.get("filename", "unknown") for doc in all_documents})
    print(f"\nUnique source files ({len(unique_files)}):")
    for fname in unique_files:
        print(f"  - {fname}")


Document Loading Summary
PDF pages         : 12
DOCX pseudo-pages : 3
TXT files         : 1
--------------------------------------------------
Total documents merged: 16

Unique source files (5):
  - AI.pdf
  - Cloud_Computing.txt
  - Database.docx
  - Machine_Learning.pdf
  - Python.pdf


## Cell 9 — Chunk Documents

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------------------
# Bonus: adjustable chunking parameters.
# Change these values and re-run Cells 9-12 to rebuild the index
# with a different chunk size / overlap.
# ---------------------------------------------------------------
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# split_documents() automatically propagates each source Document's
# metadata (filename, page, doc_type) onto every chunk it produces.
document_chunks = text_splitter.split_documents(all_documents)

# Attach a unique chunk_id to each chunk for traceability/debugging.
for idx, chunk in enumerate(document_chunks):
    chunk.metadata["chunk_id"] = idx

print(f"Original documents : {len(all_documents)}")
print(f"Total chunks created: {len(document_chunks)}")
print(f"Chunk size          : {CHUNK_SIZE}")
print(f"Chunk overlap        : {CHUNK_OVERLAP}")

if document_chunks:
    print("\nSample chunk metadata:")
    print(document_chunks[0].metadata)


Original documents : 16
Total chunks created: 66
Chunk size          : 1000
Chunk overlap        : 150

Sample chunk metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-29T09:03:51+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-29T09:03:51+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'docs/AI.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'filename': 'AI.pdf', 'doc_type': 'pdf', 'chunk_id': 0}


## Cell 10 — Create Embeddings

In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Initialize the embedding model. encode_kwargs configures batched
# encoding under the hood (sentence-transformers handles batching
# internally when embedding many texts at once).
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,  # Batch embedding for efficiency on larger document sets
    },
)

try:
    if not document_chunks:
        raise ValueError("No document chunks available. Run Cells 5-9 first.")

    # Sanity check: embed a single sample chunk to confirm the model works
    # and to report the embedding vector dimension.
    sample_vector = embedding_model.embed_query(document_chunks[0].page_content)

    print(f"Embedding model loaded: {EMBEDDING_MODEL_NAME}")
    print(f"Embedding vector dimension: {len(sample_vector)}")
    print(f"Total chunks ready to embed: {len(document_chunks)}")

except Exception as e:
    print(f"Error initializing embedding model: {e}")


/tmp/ipykernel_2212/1109531339.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding vector dimension: 384
Total chunks ready to embed: 66


## Cell 11 — Build FAISS Index

In [16]:
from langchain_community.vectorstores import FAISS

try:
    if not document_chunks:
        raise ValueError("No document chunks available. Run Cells 5-9 first.")

    # FAISS.from_documents embeds every chunk (in batches, via embedding_model)
    # and builds an in-memory FAISS index in a single call.
    vectorstore = FAISS.from_documents(document_chunks, embedding_model)

    print("FAISS vector store built successfully.")
    print(f"Total vectors stored in index: {vectorstore.index.ntotal}")

except Exception as e:
    print(f"Error building FAISS index: {e}")


FAISS vector store built successfully.
Total vectors stored in index: 66


## Cell 12 — Save FAISS Index (Persist to Disk)

In [17]:
import os

VECTORSTORE_DIR = "vectorstore"
os.makedirs(VECTORSTORE_DIR, exist_ok=True)

try:
    vectorstore.save_local(VECTORSTORE_DIR)
    print(f"FAISS index persisted to disk at: '{VECTORSTORE_DIR}/'")
    print("Files saved:", sorted(os.listdir(VECTORSTORE_DIR)))
except Exception as e:
    print(f"Error saving FAISS index: {e}")

print("\nIndexing pipeline complete. You may now restart the runtime and")
print("run only Cells 1, 3, 4, and 13-18 to query this saved index later.")


FAISS index persisted to disk at: 'vectorstore/'
Files saved: ['index.faiss', 'index.pkl']

Indexing pipeline complete. You may now restart the runtime and
run only Cells 1, 3, 4, and 13-18 to query this saved index later.


## Cell 13 — Load FAISS Index (Start of the Querying Pipeline)

This cell is intentionally **self-contained**: it re-initializes the embedding model and
loads the FAISS index straight from disk. This means you can restart the notebook and run
only Cells 1, 3, 4, 13-18 to query a previously built index, without redoing the
indexing steps (Cells 5-12).

In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTORSTORE_DIR = "vectorstore"

# Re-initialize the embedding model (safe to re-run even if already defined earlier)
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

try:
    vectorstore = FAISS.load_local(
        VECTORSTORE_DIR,
        embedding_model,
        allow_dangerous_deserialization=True,  # Safe: we created this index ourselves
    )
    print(f"FAISS index loaded successfully from '{VECTORSTORE_DIR}/'.")
    print(f"Total vectors in index: {vectorstore.index.ntotal}")

except Exception as e:
    print(f"Error loading FAISS index: {e}")
    print("Make sure you have run the indexing cells (Cells 5-12) at least once.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index loaded successfully from 'vectorstore/'.
Total vectors in index: 66


## Cell 14 — Retriever

In [19]:
# ---------------------------------------------------------------
# Bonus: adjustable top_k. Change this value and re-run this cell
# (and Cell 16) to retrieve a different number of chunks per query.
# ---------------------------------------------------------------
TOP_K = 4

try:
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
    print(f"Retriever initialized successfully with top_k = {TOP_K}")
except Exception as e:
    print(f"Error creating retriever: {e}")


Retriever initialized successfully with top_k = 4


## Cell 15 — Prompt Template (Strict, No Hallucination)

In [20]:
from langchain_core.prompts import PromptTemplate

FALLBACK_MESSAGE = "I couldn't find this information in the uploaded documents."

qa_prompt_template = (
    "You are a precise, factual assistant that answers questions using ONLY the "
    "context provided below.\n\n"
    "Strict rules you must follow:\n"
    "1. Use ONLY the information contained in the context to answer the question.\n"
    "2. Do NOT use any outside knowledge, do NOT make assumptions, and do NOT guess.\n"
    "3. If the context does not contain enough information to answer the question, "
    "respond with EXACTLY this sentence and nothing else:\n"
    f"   \"{FALLBACK_MESSAGE}\"\n"
    "4. Be concise, accurate, and do not fabricate facts, names, or numbers.\n\n"
    "Context:\n"
    "{context}\n\n"
    "Question:\n"
    "{question}\n\n"
    "Answer:"
)

qa_prompt = PromptTemplate(
    template=qa_prompt_template,
    input_variables=["context", "question"],
)

print("Strict, no-hallucination prompt template created.")
print(f"Fallback message: \"{FALLBACK_MESSAGE}\"")


Strict, no-hallucination prompt template created.
Fallback message: "I couldn't find this information in the uploaded documents."


## Cell 16 — RetrievalQA Chain

In [21]:
from langchain_classic.chains import RetrievalQA

try:
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        return_source_documents=True,
        chain_type_kwargs={"prompt": qa_prompt},
    )
    print("RetrievalQA chain created successfully.")
    print(f"Using top_k = {TOP_K} retrieved chunks per query.")
    print(f"Using Gemini model: {GEMINI_MODEL_NAME}")
except Exception as e:
    print(f"Error creating RetrievalQA chain: {e}")


RetrievalQA chain created successfully.
Using top_k = 4 retrieved chunks per query.
Using Gemini model: gemini-2.5-flash


## Cell 17 — Interactive Question Loop

In [ ]:
def format_citations(source_documents):
    """
    Build a de-duplicated list of citation strings ("filename Page N")
    from a list of retrieved source Document objects.
    """
    citations = []
    seen = set()
    for doc in source_documents:
        filename = doc.metadata.get("filename", "Unknown")
        page = doc.metadata.get("page", "N/A")
        citation_key = f"{filename}|{page}"
        if citation_key not in seen:
            citations.append(f"{filename} Page {page}")
            seen.add(citation_key)
    return citations


def ask_question(question):
    """Run a single question through the RetrievalQA chain with error handling."""
    try:
        result = qa_chain.invoke({"query": question})
        return result
    except Exception as e:
        print(f"Error while processing the question: {e}")
        return None


print("=" * 60)
print("Document Q&A Bot — Interactive Mode")
print("Type 'exit' or 'quit' (or leave the input empty) to stop.")
print("=" * 60)

while True:
    user_question = input("\nYour question: ").strip()

    if user_question.lower() in ("exit", "quit", ""):
        print("Ending Q&A session.")
        break

    result = ask_question(user_question)
    if result is None:
        continue

    answer = result.get("result", "").strip()
    source_docs = result.get("source_documents", [])

    print("\nAnswer:")
    print(answer)

    if "couldn\'t find this information" not in answer.lower():
        citations = format_citations(source_docs)
        if citations:
            print("\nSource:")
            for citation in citations:
                print(f"  - {citation}")


Document Q&A Bot — Interactive Mode
Type 'exit' or 'quit' (or leave the input empty) to stop.

Your question: what is machine learning?

Answer:
Machine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly programmed. ML algorithms build models from training data and use those models to make predictions or decisions.

Source:
  - AI.pdf Page 0
  - Machine_Learning.pdf Page 0
  - AI.pdf Page 1

Your question: What is the difference between supervised and unsupervised learning?

Answer:
I couldn't find this information in the uploaded documents.

Your question: what is meant by cloude computing?

Answer:
Cloud computing is the delivery of computing services—including servers, storage, databases, networking, software, analytics, and intelligence—over the internet ("the cloud") to offer faster innovation, flexible resources, and economies of scale. Users typically pay only for the cloud services they use, helping them lower operating costs, run i

In [23]:
# ─── Retrieval Diagnostic ─────────────────────────────────────────────────────
from collections import Counter

# 1. Check what source files are actually IN the vectorstore
print("=== Files indexed in vectorstore ===")
all_meta = vectorstore.docstore._dict.values()
file_counts = Counter(doc.metadata.get("filename", "unknown") for doc in all_meta)
for fname, count in sorted(file_counts.items()):
    print(f"  {count:>4} chunks  →  {fname}")

# 2. Spot-check retrieval per document
print("\n=== Retrieval test per topic ===")
test_queries = {
    "ML (supervised)"  : "supervised learning algorithms classification regression",
    "ML (unsupervised)": "clustering k-means unsupervised learning",
    "Database"         : "SQL database ACID transactions NoSQL",
    "Cloud"            : "IaaS PaaS SaaS cloud computing AWS",
    "Python"           : "Python object oriented programming class inheritance",
    "AI"               : "neural network deep learning transformer attention",
}
for label, q in test_queries.items():
    hits = vectorstore.similarity_search(q, k=3)
    sources = [h.metadata.get("filename", "?") for h in hits]
    print(f"  {label:<22} → {sources}")

=== Files indexed in vectorstore ===
    11 chunks  →  AI.pdf
    23 chunks  →  Cloud_Computing.txt
    10 chunks  →  Database.docx
    11 chunks  →  Machine_Learning.pdf
    11 chunks  →  Python.pdf

=== Retrieval test per topic ===
  ML (supervised)        → ['Machine_Learning.pdf', 'AI.pdf', 'Machine_Learning.pdf']
  ML (unsupervised)      → ['Machine_Learning.pdf', 'Machine_Learning.pdf', 'Machine_Learning.pdf']
  Database               → ['Database.docx', 'Database.docx', 'Database.docx']
  Cloud                  → ['Cloud_Computing.txt', 'Cloud_Computing.txt', 'Cloud_Computing.txt']
  Python                 → ['Python.pdf', 'Python.pdf', 'Python.pdf']
  AI                     → ['AI.pdf', 'AI.pdf', 'Machine_Learning.pdf']


## Cell 18 — Pretty Output With Citations (+ Bonus Features)

In [ ]:
from langchain_classic.chains import RetrievalQA


def display_answer_with_citations(question, top_k=None, show_chunks=True):
    """
    Run a single query end-to-end and print a nicely formatted report:
      - The Gemini-generated answer (grounded only in retrieved context)
      - De-duplicated citations (filename + page)
      - Bonus: the raw retrieved chunks with their FAISS similarity scores

    Parameters
    ----------
    question : str
        The natural-language question to ask.
    top_k : int, optional
        Override TOP_K for this call only (bonus: change top_k per query
        without rebuilding the whole chain). Defaults to the global TOP_K.
    show_chunks : bool
        Whether to print the retrieved chunks and similarity scores.
    """
    k = top_k if top_k is not None else TOP_K

    # Build a retriever + chain scoped to this call\'s top_k value
    local_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    local_qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=local_retriever,
        chain_type="stuff",
        return_source_documents=True,
        chain_type_kwargs={"prompt": qa_prompt},
    )

    try:
        result = local_qa_chain.invoke({"query": question})
    except Exception as e:
        print(f"Error while processing the question: {e}")
        return None

    answer = result.get("result", "").strip()
    source_docs = result.get("source_documents", [])

    print("┌" + "─" * 58 + "┐")
    print(f"Question: {question}")
    print("└" + "─" * 58 + "┘")
    print("\nAnswer:")
    print(answer)

    if "couldn\'t find this information" not in answer.lower():
        print("\nSource:")
        for citation in format_citations(source_docs):
            print(f"  - {citation}")

    if show_chunks:
        print("\n--- Retrieved Chunks (bonus: with similarity scores) ---")
        try:
            scored_chunks = vectorstore.similarity_search_with_score(question, k=k)
            for idx, (doc, score) in enumerate(scored_chunks, start=1):
                filename = doc.metadata.get("filename", "Unknown")
                page = doc.metadata.get("page", "N/A")
                # FAISS returns L2 distance by default: lower score = more similar
                print(f"\n[{idx}] {filename} | Page {page} | Similarity score (lower = closer): {score:.4f}")
                preview = doc.page_content[:300].replace("\n", " ")
                print(f"    Preview: {preview}...")
        except Exception as e:
            print(f"Error retrieving chunk previews: {e}")

    return result


# ---------------------------------------------------------------
# Example usage / demo run
# Change `sample_question` and `top_k` below to experiment.
# ---------------------------------------------------------------
sample_question = "What is this document about?"
_ = display_answer_with_citations(sample_question, top_k=TOP_K, show_chunks=True)


┌──────────────────────────────────────────────────────────┐
Question: What is this document about?
└──────────────────────────────────────────────────────────┘

Answer:
This document is about Python programming, covering its introduction, definition, creation, features, data types, functions, and popular use cases.

Source:
  - Unknown Page 0
  - Unknown Page 2
  - Unknown Page 1
  - Unknown Page 4

--- Retrieved Chunks (bonus: with similarity scores) ---

[1] Unknown | Page 0 | Similarity score (lower = closer): 1.6627
    Preview: Introduction to Python Programming Chapter 1: What is Python? Python is a high-level, interpreted programming language known for its clear, readable syntax and broad standard library. It was created by Guido van Rossum and was first released in the year 1991. Python supports multiple programming par...

[2] Unknown | Page 2 | Similarity score (lower = closer): 1.7792
    Preview: Chapter 3: Functions in Python A function in Python is defined using the 'def